# Packages

In [1]:
import sys
sys.path.append('/home/boscoll/github/ReCAST/code')
from ReCAST import ReCAST
import matplotlib as mpl
import pickle
import pandas as pd
from pathlib import Path

# Data load

In [2]:
project_path = Path('/home/boscoll/Projects/cdk_predict')
train_test_split = project_path / 'data/development_cohort/train_test_split/genomic_train_test_split_msk_final'
clinical_path = project_path / 'data/development_cohort/clinical/clinical_msk_final.csv'


with open(train_test_split, 'rb') as f:
    X_genomic_train, X_genomic_test, y_train, y_test = pickle.load(f)
y_train = y_train[['Time', 'Event']].copy()

clinical = pd.read_csv(clinical_path, index_col=0)

#merged clinicogenomic
X_clinicogenomic_train = pd.concat([X_genomic_train, clinical.loc[lambda x: x.index.isin(X_genomic_train.index)]], axis=1)
X_clinicogenomic_test = pd.concat([X_genomic_test, clinical.loc[lambda x: x.index.isin(X_genomic_test.index)]], axis=1)
X_clinicogenomic_train = X_clinicogenomic_train.reindex(y_train.index)
X_clinicogenomic_test = X_clinicogenomic_test.reindex(y_test.index)

#from the clinical only dataframe, remove the 'met_sample' variable
X_clinical_train = clinical.loc[X_genomic_train.index].copy()
X_clinical_test = clinical.loc[X_genomic_test.index].copy()
X_clinical_train.drop(columns='met_sample', inplace=True)
X_clinical_test.drop(columns='met_sample', inplace=True)

## Clinical only ReCAST fine-tuned on training set

In [3]:
#lasso framework
ReCAST_clin = ReCAST(
    n_models = 100, 
    l1 = 1, 
    n_folds=3, 
    bootstrap=True,
    random_state=94, 
    adaptive_lasso=True, 
    normalization = (0.01, 0.99), 
    metric = 'auc', 
    auc_time_range =(3, 36), 
)
ReCAST_clin.fit(X_clinical_train, y_train)

Fitting 100 CoxNet Models...
Selected `bootstrap` sampling for model fitting
Selected the option `adaptive_lasso` to assign distinct penalty to different coefficients
Using time-dependent AUC as model selection metric...
Using user-defined AUC time range for hyperparameter tuning: (3, 36)
Finished fitting 100 models.


In [ ]:
# ReCAST_clin.save_model('/home/boscoll/Projects/cdk_predict/results/models_selection/models/ReCAST_clin_train_final.pkl')

Model saved to /home/boscoll/Projects/cdk_predict/results/models_selection/models/ReCAST_clin_train_final.pkl


## Genomic only ReCAST fine-tuned on training set

In [5]:
#lasso framework
ReCAST_genomics = ReCAST(
    n_models = 100, 
    l1 = 1, 
    n_folds=3, 
    bootstrap=True,
    random_state=94, 
    adaptive_lasso=True, 
    normalization=(0.01, 0.99), 
    auc_time_range=(3, 36),
    metric = 'auc'
)
ReCAST_genomics.fit(X_genomic_train, y_train)


Fitting 100 CoxNet Models...
Selected `bootstrap` sampling for model fitting
Selected the option `adaptive_lasso` to assign distinct penalty to different coefficients
Using time-dependent AUC as model selection metric...
Using user-defined AUC time range for hyperparameter tuning: (3, 36)
Finished fitting 100 models.


In [ ]:
# ReCAST_genomics.save_model('/home/boscoll/Projects/cdk_predict/results/models_selection/models/ReCAST_genomics_train_final.pkl')

Model saved to /home/boscoll/Projects/cdk_predict/results/models_selection/models/ReCAST_genomics_train_final.pkl


## CDKPredict fine-tuned on training set

In [7]:
#lasso framework
CDKPredict = ReCAST(
    n_models = 100, 
    l1 = 1,
    val_size=0.3, 
    n_folds=3, 
    bootstrap=True,
    random_state=94, 
    adaptive_lasso=True,                
    normalization=(0.01, 0.99), 
    auc_time_range=(3, 36),
    metric = 'auc'
)
CDKPredict.fit(X_clinicogenomic_train, y_train)

Fitting 100 CoxNet Models...
Selected `bootstrap` sampling for model fitting
Selected the option `adaptive_lasso` to assign distinct penalty to different coefficients
Using time-dependent AUC as model selection metric...
Using user-defined AUC time range for hyperparameter tuning: (3, 36)
Finished fitting 100 models.


In [8]:
CDKPredict.save_model('/home/boscoll/Projects/cdk_predict/results/models_selection/models/CDKPredict_train_final.pkl')

Model saved to /home/boscoll/Projects/cdk_predict/results/models_selection/models/CDKPredict_train_final.pkl


## CDKPredict fine-tuned on overall MSK-cohort before model deployment

In [9]:
y_full = pd.concat([y_train, y_test], axis=0)
X_full_clinicogenomic = pd.concat([X_clinicogenomic_train, X_clinicogenomic_test])
CDKPredict_final = ReCAST(
    n_models = 100, 
    l1 = 1,
    n_folds=3, 
    bootstrap=True,
    random_state=94, 
    adaptive_lasso=True, 
    normalization=(0.01, 0.99), 
    auc_time_range=(3, 36),
    metric = 'auc', 
)
CDKPredict_final.fit(X_full_clinicogenomic, y_full)

Fitting 100 CoxNet Models...
Selected `bootstrap` sampling for model fitting
Selected the option `adaptive_lasso` to assign distinct penalty to different coefficients
Using time-dependent AUC as model selection metric...
Using user-defined AUC time range for hyperparameter tuning: (3, 36)
Finished fitting 100 models.


In [ ]:
# CDKPredict_final.save_model('/home/boscoll/Projects/cdk_predict/results/models_selection/models/CDKPredict_msk_final.pkl')

Model saved to /home/boscoll/Projects/cdk_predict/results/models_selection/models/CDKPredict_msk_final.pkl
